# Notebook 2 — Pre-clasificacion Automatica + UI de Etiquetado

1. **Pre-clasifica** todos los segmentos con heuristicas espectrales + PANNs (AudioSet).
2. **Interfaz Gradio** para revision humana: reproducis el audio, ves el espectrograma,
   y aprobas o corregis la prediccion. El progreso se guarda automaticamente.

In [1]:
# CELDA 1 — CONFIGURACION + AUTO-DIMENSIONAMIENTO SEGURO DE RECURSOS
import os
import math
import warnings
import psutil
import torch
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

METADATA_PATH = r"C:\temporales\metadata.csv"
SPEC_TEMP_DIR = r"C:\temporales\specs"
MODEL_PATH    = r"C:\temporales\modelos\clasificador_radio_v1.pth"

CATEGORIAS = [
    "musica", "locutor", "deportes", "entrevistas", "indefinido"
]

# Confianza >= este valor -> auto-etiquetado sin revision humana
AUTO_LABEL_THRESHOLD = 0.90
# Confianza < este valor -> aparece PRIMERO en la cola de revision
LOW_CONF_THRESHOLD   = 0.60

# ── AUTO-DIMENSIONAMIENTO SEGURO Y CONSERVADOR (RAM / VRAM) ──────────────
# 1. RAM: Se calcula CHUNK_SIZE sobre la RAM disponible (máx 4096 clips por bloque = ~12 GB RAM)
avail_ram_bytes  = psutil.virtual_memory().available
ram_target_bytes = avail_ram_bytes * 0.50  # Usar máximo el 50% de la RAM DISPONIBLE
bytes_per_clip   = 3.5 * 1024 * 1024       # ~3.5 MB por clip en RAM (arrays, np.stack, overhead Python)
CHUNK_SIZE       = min(4096, max(512, int(ram_target_bytes / bytes_per_clip)))

# 2. VRAM GPU: BATCH_SIZE seguro para PANNs CNN14 (evita picos por activaciones convolucionales)
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb >= 16:
        PANNS_BATCH_SIZE = 256  # Súper estable y llena los Tensor Cores sin saturar VRAM
    elif vram_gb >= 8:
        PANNS_BATCH_SIZE = 128
    else:
        PANNS_BATCH_SIZE = 64
else:
    PANNS_BATCH_SIZE = 64

os.makedirs(SPEC_TEMP_DIR, exist_ok=True)

print("[i] Configuración y Auto-dimensionamiento Seguro de Recursos:")
print(f"   RAM Disponible   : {avail_ram_bytes / (1024**3):.1f} GB (Objetivo = {ram_target_bytes / (1024**3):.1f} GB)")
print(f"   CHUNK_SIZE auto  : {CHUNK_SIZE:,} clips por bloque (~{(CHUNK_SIZE * bytes_per_clip) / (1024**3):.2f} GB RAM máx)")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"   GPU Detectada     : {gpu_name} ({vram_total:.1f} GB VRAM)")
    print(f"   PANNS_BATCH_SIZE : {PANNS_BATCH_SIZE} por lote GPU")
else:
    print("   GPU              : No disponible (CPU fallback)")


[i] Configuración y Auto-dimensionamiento Seguro de Recursos:
   RAM Disponible   : 14.0 GB (Objetivo = 7.0 GB)
   CHUNK_SIZE auto  : 2,051 clips por bloque (~7.01 GB RAM máx)
   GPU Detectada     : NVIDIA GeForce RTX 3090 (24.0 GB VRAM)
   PANNS_BATCH_SIZE : 256 por lote GPU


In [2]:
# CELDA 3 — CARGAR MODELO PANNS (AUDIOSET MAP EXPANDIDO)
AUDIOSET_MAP = {
    # ── MÚSICA ───────────────────────────────────────────────────
    'Music': 'musica', 'Musical': 'musica', 'Singing': 'musica',
    'Song': 'musica', 'Jingle': 'musica', 'Ringtone': 'musica',
    'Tune': 'musica', 'Melody': 'musica', 'Choir': 'musica',
    'Guitar': 'musica', 'Piano': 'musica', 'Drum': 'musica',
    'Brass': 'musica', 'Saxophone': 'musica', 'Violin': 'musica',
    'Synthesizer': 'musica', 'Accordion': 'musica', 'Organ': 'musica',
    'Trumpet': 'musica', 'Bass': 'musica', 'Keyboard': 'musica',
    'Pop': 'musica', 'Rock': 'musica', 'Jazz': 'musica',
    'Salsa': 'musica', 'Techno': 'musica', 'Disco': 'musica',
    'Tango': 'musica', 'Folk': 'musica', 'Orchestra': 'musica',

    # ── LOCUTOR (Voz hablada individual / Presentador) ───────────
    'Speech': 'locutor', 'Narration': 'locutor', 'Monologue': 'locutor',
    'Announcement': 'locutor', 'Presenter': 'locutor', 'Voice': 'locutor',
    'Whispering': 'locutor', 'Radio': 'locutor', 'Public address': 'locutor',

    # ── ENTREVISTAS (Diálogos / Conversación / Debate) ────────────
    'Conversation': 'entrevistas', 'Discussion': 'entrevistas',
    'Interview': 'entrevistas', 'Debate': 'entrevistas',
    'Chatter': 'entrevistas', 'Babble': 'entrevistas', 'Dialogue': 'entrevistas',

    # ── DEPORTES (Estadio / Ovaciones / Silbatos / Relatos) ──────
    'Sports': 'deportes', 'Crowd': 'deportes', 'Cheering': 'deportes',
    'Applause': 'deportes', 'Applauding': 'deportes', 'Clapping': 'deportes',
    'Stadium': 'deportes', 'Arena': 'deportes', 'Whistle': 'deportes',
    'Referee': 'deportes', 'Audience': 'deportes', 'Buzzer': 'deportes',
    'Screaming': 'deportes',

    # ── INDEFINIDO (Silencio / Estática / Ruidos de fondo) ─────────
    'Silence': 'indefinido', 'Noise': 'indefinido', 'Static': 'indefinido',
    'Hum': 'indefinido', 'Hiss': 'indefinido', 'Distortion': 'indefinido',
    'Interference': 'indefinido', 'Background noise': 'indefinido',
    'Cough': 'indefinido', 'Breathe': 'indefinido', 'Siren': 'indefinido',
}

def map_audioset(label_scores):
    best_cat, best_score = 'indefinido', 0.0
    for label, score in label_scores[:5]:
        for kw, cat in AUDIOSET_MAP.items():
            if kw.lower() in label.lower() and score > best_score:
                best_score = score; best_cat = cat; break
    return best_cat, round(float(best_score), 3)

print("Cargando modelo PANNs CNN14...")
try:
    import torch
    from panns_inference import AudioTagging
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    panns_model = AudioTagging(checkpoint_path=None, device=device)
    USE_PANNS = True
    USE_HEUR = False
    print(f"Modelo PANNs cargado ({device}) | Mapa AudioSet expandido a {len(AUDIOSET_MAP)} reglas")
except Exception as e:
    print(f"PANNs no disponible: {e}")
    print("Se usaran solo heuristicas espectrales.")
    USE_PANNS = False


Cargando modelo PANNs CNN14...
Checkpoint path: C:\Users\sukho/panns_data/Cnn14_mAP=0.431.pth
GPU number: 1
Modelo PANNs cargado (cuda) | Mapa AudioSet expandido a 70 reglas


In [3]:
# CELDA 4 — PRE-CLASIFICAR (Modelo Propio Entrenado o Fallback Heuristicas + PANNs)
import soundfile as sf
import concurrent.futures
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import timm
import librosa
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

df = pd.read_csv(METADATA_PATH)

# Clasificar segmentos no validados por humano
to_classify_mask = (df['label_fuente'] != 'humano')
sin_label_indices = df[to_classify_mask].index.tolist()
print(f"Total segmentos en metadata      : {len(df):,}")
print(f"Validados por humano (no tocan)  : {(df['label_fuente'] == 'humano').sum():,}")
print(f"Pendientes de pre-clasificar     : {len(sin_label_indices):,}")

if Path(MODEL_PATH).exists() and len(sin_label_indices) > 0:
    print(f"\n[★] Modelo propio entrenado encontrado: {MODEL_PATH}")
    print("    Pre-clasificando con modelo propio (EfficientNet-B0 GPU)...")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint   = torch.load(MODEL_PATH, map_location=device)
    LABEL_TO_IDX = checkpoint['label_to_idx']
    IDX_TO_LABEL = checkpoint['idx_to_label']
    cfg          = checkpoint['config']
    N_CLASES     = len(LABEL_TO_IDX)

    SAMPLE_RATE = cfg.get('sample_rate', 32000)
    N_MELS      = cfg.get('n_mels', 128)
    N_FFT       = cfg.get('n_fft', 2048)
    HOP_LENGTH  = cfg.get('hop_length', 1024)
    IMG_SIZE    = cfg.get('img_size', 224)

    model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=N_CLASES)
    model.load_state_dict(checkpoint['model_state'])
    model = model.to(device); model.eval()

    class NB02InferenceDataset(Dataset):
        def __init__(self, paths):
            self.paths = paths
            self.mel_t = T.MelSpectrogram(
                sample_rate=SAMPLE_RATE, n_fft=N_FFT,
                hop_length=HOP_LENGTH, n_mels=N_MELS, f_max=15000)
            self.db_t = T.AmplitudeToDB(top_db=80)

        def __len__(self): return len(self.paths)

        def __getitem__(self, idx):
            try:
                wf, sr = torchaudio.load(self.paths[idx])
                if sr != SAMPLE_RATE: wf = T.Resample(sr, SAMPLE_RATE)(wf)
                if wf.shape[0] > 1:  wf = wf.mean(dim=0, keepdim=True)
                mel = self.db_t(self.mel_t(wf))
                mn, mx = mel.min(), mel.max()
                if mx > mn: mel = 2.0 * (mel - mn) / (mx - mn) - 1.0
                mel_r = torch.nn.functional.interpolate(
                    mel.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE),
                    mode='bilinear', align_corners=False).squeeze(0)
                return mel_r.repeat(3, 1, 1)
            except Exception:
                return torch.zeros(3, IMG_SIZE, IMG_SIZE)

    wav_paths = df.loc[sin_label_indices, 'ruta_wav'].tolist()
    inf_ds = NB02InferenceDataset(wav_paths)
    inf_loader = DataLoader(inf_ds, batch_size=128, shuffle=False, num_workers=0)

    all_preds, all_confs = [], []
    with torch.no_grad():
        for batch in tqdm(inf_loader, desc="Modelo propio GPU", unit="batch"):
            batch = batch.to(device, non_blocking=True)
            with autocast():
                logits = model(batch)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_preds.extend([IDX_TO_LABEL[i] for i in probs.argmax(axis=1)])
            all_confs.extend(probs.max(axis=1).tolist())

    df.loc[sin_label_indices, 'label']        = all_preds
    df.loc[sin_label_indices, 'confianza']    = all_confs
    df.loc[sin_label_indices, 'label_fuente'] = 'modelo_propio'
    df.loc[sin_label_indices, 'revisado']     = [c >= AUTO_LABEL_THRESHOLD for c in all_confs]

    df.to_csv(METADATA_PATH, index=False)
    print("\nPre-clasificación con modelo propio completada y guardada.")
    print(df.groupby('label')['label'].count().rename('cantidad').to_frame())
    print(f"Auto-aprobados (confianza >= {AUTO_LABEL_THRESHOLD:.0%}): {(df['revisado'] & (df['label_fuente'] == 'modelo_propio')).sum():,}")
    print(f"Pendientes de revision en Gradio: {(~df['revisado']).sum():,}")

else:
    if not Path(MODEL_PATH).exists():
        print("\n[i] No hay modelo propio entrenado todavia.")
    print("    Usando pipeline de fallback (Heuristicas espectrales + PANNs)...\n")

    USE_HEUR          = False  # True = usar heurísticas de filtrado primero | False = mandar todo directo a PANNs GPU
    HEURISTIC_WORKERS = 24
    PANNS_BATCH_SIZE  = globals().get('PANNS_BATCH_SIZE', 128)
    PANNS_SR          = 32_000
    MAX_SAMPLES       = PANNS_SR * 10  # Ajustado a 10s (todos los segmentos miden 10.0s exactos)

    sin_label = df[df['label'].isna() | (df['label_fuente'] != 'humano')].copy().reset_index(drop=True)
    wav_paths = sin_label['ruta_wav'].tolist()

    from _heur_worker import extract_and_classify

    final_results = {}
    panns_indices = []

    ex_heur = concurrent.futures.ProcessPoolExecutor(max_workers=HEURISTIC_WORKERS)
    ex_io   = concurrent.futures.ThreadPoolExecutor(max_workers=HEURISTIC_WORKERS)

    try:
        if USE_HEUR:
            print(f"Fase 1 — heuristicas en {HEURISTIC_WORKERS} procesos ({len(wav_paths):,} segmentos)...")
            futures_heur = {ex_heur.submit(extract_and_classify, path): i for i, path in enumerate(wav_paths)}

            for future in tqdm(concurrent.futures.as_completed(futures_heur), total=len(futures_heur), desc="Heuristicas", unit="seg"):
                i = futures_heur[future]
                try:
                    lbl, conf, fuente = future.result()
                    if fuente == 'error' or (lbl == 'musica' and conf >= 0.80):
                        final_results[i] = (lbl or 'indefinido', conf, fuente)
                    else:
                        panns_indices.append((i, lbl, conf))
                except Exception:
                    final_results[i] = ('indefinido', 0.0, 'error')
            print(f"  Alta confianza heuristica : {len(final_results):,}")
        else:
            print("Fase 1 — Heurísticas desactivadas (USE_HEUR=False). Enviando todos los segmentos a PANNs...")
            panns_indices = [(i, None, 0.0) for i in range(len(wav_paths))]
            
        print(f"  Pasan a PANNs             : {len(panns_indices):,}")

        def load_for_panns(args):
            i, lbl_h, conf_h = args
            row = sin_label.iloc[i]
            try:
                audio, sr = sf.read(row['ruta_wav'], dtype='float32')
                if audio.ndim > 1: audio = audio.mean(axis=1)
                if sr != PANNS_SR: audio = librosa.resample(audio, orig_sr=sr, target_sr=PANNS_SR)
                audio = audio[:MAX_SAMPLES] if len(audio) >= MAX_SAMPLES else np.pad(audio, (0, MAX_SAMPLES - len(audio)))
                return i, lbl_h, conf_h, audio
            except Exception:
                return i, lbl_h, conf_h, None

        if USE_PANNS and panns_indices:
            import gc, psutil
            from panns_inference import labels as asl
            CHUNK_SIZE = globals().get('CHUNK_SIZE', 1024)
            print(f"Fase 2 — PANNs GPU en streaming por chunks de {CHUNK_SIZE} (Guardia RAM <= 80%)...")

            for chunk_start in tqdm(range(0, len(panns_indices), CHUNK_SIZE), desc="PANNs Chunks", unit="chunk"):
                if psutil.virtual_memory().percent > 80.0:
                    print(f"\n[⚠️] Uso de RAM elevado ({psutil.virtual_memory().percent:.1f}%). Liberando memoria...")
                    gc.collect()
                    if torch.cuda.is_available(): torch.cuda.empty_cache()

                chunk_args = panns_indices[chunk_start : chunk_start + CHUNK_SIZE]
                futures_io = {ex_io.submit(load_for_panns, args): args for args in chunk_args}
                loaded_chunk = [future.result() for future in concurrent.futures.as_completed(futures_io)]

                for batch_start in range(0, len(loaded_chunk), PANNS_BATCH_SIZE):
                    batch = loaded_chunk[batch_start : batch_start + PANNS_BATCH_SIZE]
                    valid = [(i, lh, ch, a) for i, lh, ch, a in batch if a is not None]
                    invalid = [(i, lh, ch) for i, lh, ch, a in batch if a is None]

                    for i, lh, ch in invalid: final_results[i] = (lh or 'indefinido', ch, 'error')
                    if not valid: continue

                    audio_batch = np.stack([a for *_, a in valid])
                    with torch.no_grad(): clip_out, _ = panns_model.inference(audio_batch)

                    for (i, lbl_h, conf_h, _), clip_scores in zip(valid, clip_out):
                        top_idx = np.argsort(clip_scores)[::-1][:10]
                        label_scores = [(asl[j], float(clip_scores[j])) for j in top_idx]
                        lbl_p, conf_p = map_audioset(label_scores)

                        if lbl_h == 'musica' and lbl_p == 'musica':
                            final_results[i] = ('musica', min(0.97, (conf_h + conf_p) / 2 + 0.1), 'panns')
                        else:
                            final_results[i] = (lbl_p, conf_p, 'panns')

                del loaded_chunk, futures_io
                gc.collect()
        else:
            for i, lbl_h, conf_h in panns_indices:
                final_results[i] = (lbl_h or 'indefinido', conf_h, 'heuristica')

    except KeyboardInterrupt:
        print("\n[!] INTERRUMPIDO POR EL USUARIO.")
        ex_heur.shutdown(wait=False, cancel_futures=True)
        ex_io.shutdown(wait=False, cancel_futures=True)
    except Exception as e:
        print(f"\n[!] ERROR INESPERADO: {e}")
        ex_heur.shutdown(wait=False, cancel_futures=True)
        ex_io.shutdown(wait=False, cancel_futures=True)
    else:
        ex_heur.shutdown(wait=True)
        ex_io.shutdown(wait=True)

    procesados_idx = list(final_results.keys())
    if procesados_idx:
        labels_auto, confs_auto, fuentes_auto = [], [], []
        for i in procesados_idx:
            lbl, conf, src = final_results[i]
            labels_auto.append(lbl); confs_auto.append(conf); fuentes_auto.append(src)

        sin_label_actualizado = sin_label.loc[procesados_idx].copy()
        sin_label_actualizado['label']        = labels_auto
        sin_label_actualizado['confianza']    = confs_auto
        sin_label_actualizado['label_fuente'] = fuentes_auto
        sin_label_actualizado['revisado']     = sin_label_actualizado['confianza'] >= AUTO_LABEL_THRESHOLD

        sin_label_restantes = sin_label.drop(procesados_idx)

        df_final = pd.concat([
            df[df['label_fuente'] == 'humano'].copy(),
            sin_label_actualizado,
            sin_label_restantes
        ], ignore_index=True)

        df_final.to_csv(METADATA_PATH, index=False)
        print("Pre-clasificacion guardada.")


Total segmentos en metadata      : 46,471
Validados por humano (no tocan)  : 164
Pendientes de pre-clasificar     : 46,307

[★] Modelo propio entrenado encontrado: C:\temporales\modelos\clasificador_radio_v1.pth
    Pre-clasificando con modelo propio (EfficientNet-B0 GPU)...


Modelo propio GPU:   0%|          | 0/362 [00:00<?, ?batch/s]


Pre-clasificación con modelo propio completada y guardada.
             cantidad
label                
deportes         3487
entrevistas      2561
indefinido         36
locutor         26432
musica          13955
Auto-aprobados (confianza >= 90%): 281
Pendientes de revision en Gradio: 46,026


---
## FASE B — Interfaz de Etiquetado Humano

En esta celda haciendo se usa Gradio para tener una interfaz rapida de reproduccion de segmentos y clasificación.
Muestra los segmentos de menor a mayor confianza.
El progreso se guarda automaticamente despues de cada decisión.

In [4]:
# CELDA 5 — GENERADOR DE ESPECTROGRAMAS
import matplotlib.pyplot as plt
import librosa.display
def generar_espectrograma(wav_path, output_path, sr=32000):
    try:
        y, sr = librosa.load(wav_path, sr=sr, mono=True)
        fig, ax = plt.subplots(figsize=(10, 3))
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=15000)
        S_dB = librosa.power_to_db(S, ref=np.max)
        img = librosa.display.specshow(S_dB, sr=sr, x_axis='time',
                                       y_axis='mel', fmax=15000, ax=ax, cmap='magma')
        fig.colorbar(img, ax=ax, format='%+2.0f dB')
        ax.set_title("Mel-Espectrograma")
        plt.tight_layout()
        plt.savefig(output_path, dpi=100, bbox_inches='tight')
        plt.close(fig)
        return output_path
    except Exception:
        plt.close('all')
        return None

print("Funcion de espectrograma lista")


Funcion de espectrograma lista


In [5]:
# CELDA 6 — INTERFAZ GRADIO DE ETIQUETADO
import gradio as gr

df_meta = pd.read_csv(METADATA_PATH)
cola = df_meta[~df_meta['revisado']].sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Segmentos en cola de revision: {len(cola):,}")

estado = {"idx": 0, "guardados": 0}

def guardar_label(segment_id, label):
    df_actual = pd.read_csv(METADATA_PATH)
    df_actual.loc[df_actual['segment_id'] == segment_id, 'label']        = label
    df_actual.loc[df_actual['segment_id'] == segment_id, 'label_fuente'] = 'humano'
    df_actual.loc[df_actual['segment_id'] == segment_id, 'revisado']     = True
    df_actual.to_csv(METADATA_PATH, index=False)
    estado["guardados"] += 1

def cargar_muestra(idx):
    if idx >= len(cola):
        return None, None, "indefinido", "__fin__", "Terminaste todas las muestras!"
    row = cola.iloc[idx]
    seg_id = row['segment_id']
    label_sug = row['label'] if pd.notna(row['label']) else 'indefinido'
    confianza  = row['confianza'] if pd.notna(row['confianza']) else 0
    spec_path  = os.path.join(SPEC_TEMP_DIR, f"{seg_id}.png")
    if not Path(spec_path).exists():
        generar_espectrograma(row['ruta_wav'], spec_path)
    conf_str = "ALTA" if confianza >= 0.75 else ("MEDIA" if confianza >= 0.50 else "BAJA")
    status = (f"**Muestra {idx+1}/{len(cola)}** | "
              f"Prediccion: **{label_sug}** | "
              f"Confianza: **{confianza:.1%}** ({conf_str}) | "
              f"Guardados: **{estado['guardados']}**")
    return row['ruta_wav'], spec_path, label_sug, seg_id, status

audio0, spec0, label0, seg0, status0 = cargar_muestra(0)

with gr.Blocks(title="Radio Audio Labeler",
               theme=gr.themes.Base(primary_hue="indigo", neutral_hue="slate")) as demo:
    gr.Markdown("# Radio Audio Labeler")
    gr.Markdown("Revisa y corrige la clasificacion automatica. Progreso guardado automaticamente.")
    status_md = gr.Markdown(value=status0)
    with gr.Row():
        with gr.Column(scale=3):
            # Corrección: sin show_download_button y agregado interactive=False
            spec_img = gr.Image(value=spec0, label="Espectrograma Mel", interactive=False)
        with gr.Column(scale=2):
            audio_pl  = gr.Audio(value=audio0, label="Audio", interactive=False, autoplay=True)
            pred_box  = gr.Textbox(value=label0, label="Categoria sugerida", interactive=False)
            seg_hidden = gr.Textbox(value=seg0, visible=False)
    gr.Markdown("### Decision")
    with gr.Row():
        btn_ok   = gr.Button("Correcto", variant="primary",   size="lg")
        btn_skip = gr.Button("Saltar",   variant="secondary", size="lg")
    with gr.Row():
        corr_dd  = gr.Dropdown(choices=CATEGORIAS, label="Corregir como:",
                                value=CATEGORIAS[0])
        btn_corr = gr.Button("Corregir", variant="stop", size="lg")

    idx_state = gr.State(value=0)
    outs = [audio_pl, spec_img, pred_box, seg_hidden, status_md, idx_state]

    def ok_fn(seg_id, idx):
        guardar_label(seg_id, cola.iloc[idx]['label'])
        ni = idx + 1; return (*cargar_muestra(ni), ni)
    def corr_fn(seg_id, nueva, idx):
        guardar_label(seg_id, nueva)
        ni = idx + 1; return (*cargar_muestra(ni), ni)
    def skip_fn(idx):
        ni = idx + 1; return (*cargar_muestra(ni), ni)

    btn_ok.click(ok_fn,   [seg_hidden, idx_state], outs)
    btn_corr.click(corr_fn, [seg_hidden, corr_dd, idx_state], outs)
    btn_skip.click(skip_fn, [idx_state], outs)

demo.launch(share=False, inbrowser=True, allowed_paths=["C:/temporales"])


Segmentos en cola de revision: 46,026
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [6]:
# CELDA 7 — ESTADO DEL ETIQUETADO
df_check = pd.read_csv(METADATA_PATH)
humanos  = df_check[df_check['label_fuente'] == 'humano']
autos    = df_check[(df_check['revisado'] == True) & (df_check['label_fuente'] != 'humano')]
sin_rev  = df_check[df_check['revisado'] == False]
print(f"Etiquetados por humano : {len(humanos):>6,}")
print(f"Auto-aprobados         : {len(autos):>6,}")
print(f"Sin revisar            : {len(sin_rev):>6,}")
if len(humanos) > 0:
    print()
    print(humanos.groupby('label').size().sort_values(ascending=False).to_frame('cantidad'))


Etiquetados por humano :    164
Auto-aprobados         :    281
Sin revisar            : 46,026

             cantidad
label                
locutor            82
indefinido         36
entrevistas        23
musica             13
deportes           10
